# 10. Which TFs read a core-motif site, in which cell line (Fig. 5b,c)

Every core motif matches several candidate reader TFs (notebook 04). This notebook asks, for each cell line separately, **which of those candidate readers actually bind the motif copies that the cell line carries in its own promoter windows**, which reader is bound at the most genes, and how often the readers bound at a gene differ between two cell lines.

Five ChIP-profiled lines: K562, HepG2, A549, GM12878 and HEK293. Each line is scored on **its own** promoter windows, so a locus does not have to be present in all five; cell lines are then compared two at a time, on the loci that both lines of the pair carry.

`code/chipseq_enrichment/reader_switch/`:
1. `s1_peak_filter.py`: each TF's top-10 % peaks by signal value, within TSS +/- 10 kb, from the ENCODE download (notebook 08 explains the download; HEK293 included);
2. the **motif loci** of each line: the FIMO sites (p < 1e-4) of the 43 core motifs in that line's promoter windows, mapped to hg38. Released on Zenodo (`fig5_reader_switch_inputs.tar.gz`, `reader_switch/out/loci/{cell}.parquet`), because rebuilding them needs the promoter-window scans of the four EpiMap lines and of HEK293 (`s2_motif_loci.py`);
3. `s3c_reader_binding_percell.py`: `bedtools intersect` of each line's own loci with each candidate reader's top peaks, collapsed to one record per (cell line, gene, motif, reader);
4. `fig5bc_popularity_open.py`: **Fig. 5b** (reader-binding grids of four motifs) and **Fig. 5c** (switch rate of the 12 motifs with at least 20 gene comparisons).

This replaces the earlier build, which restricted the analysis to the loci present in all five lines at the same place and ranked readers by ChIP signal intensity; `s3_reader_binding.py` and `fig5bc_nature.py` are still in `code/` as provenance.

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/setup_work_dir.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
os.environ["ENHANCER_ROOT"], os.environ["SEQ_ROOT"] = ENHANCER_ROOT, SEQ_ROOT   # the scripts in code/ read their data root from here
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    if cmd.startswith("python "): cmd = f'"{sys.executable}" ' + cmd[7:]   # the interpreter this notebook runs on, not whatever comes first on PATH
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


In [ ]:
ENCODE = f"{ENHANCER_ROOT}/encode4_tf_chipseq"     # bin/download_encode_peaks.py --dest $ENHANCER_ROOT/encode4_tf_chipseq
def encode_status(cells):
    for c in cells:
        n = len(glob.glob(f"{ENCODE}/{c}/*/*.bed")); want = sum(1 for l in open(f"{REPO}/data/encode/encode_tf_chipseq_peaks.tsv") if l.split("\t")[0] == c)
        print(f"  {'ok ' if n == want else '-- '} {c}: {n} of {want} ENCODE peak files")
RS = f"{CODE}/chipseq_enrichment/reader_switch"; WORK = f"{ENHANCER_ROOT}/notebook_runs/fig5"; os.makedirs(WORK, exist_ok=True)
CELLS = ["K562", "HepG2", "A549", "GM12878", "HEK293"]
import numpy as np, pandas as pd
from IPython.display import Image, display
encode_status(CELLS)
print("  per-cell-line motif loci (Zenodo):", len(glob.glob(f"{RS}/out/loci/*.parquet")), "of 5 cell lines | bedtools:", shutil.which("bedtools"))

# the published tables this notebook regenerates, kept aside for comparison (the scripts overwrite them in place)
PUB = ["reader_binding_percell.parquet", "interrogated_percell.parquet", "assayed_tfs.json",
       "per_motif_setswitch_open.csv", "pairs_setswitch_open.csv"]
for f in PUB:
    src = f"{RS}/out/{f}"
    print(f"  {'ok ' if os.path.exists(src) else '-- '} published {f}")
    if os.path.exists(src): shutil.copy(src, f"{WORK}/published_{f}")

## 10.1 Top-10 % peaks of every TF (minutes)
Within each TF experiment the peaks are ranked by their own `signalValue` and the top 10 % are kept, so a weakly and a strongly sequenced experiment contribute comparable numbers of sites. `SIG_PCTL = 90` in `reader_switch/lib/config.py` sets the cut.

In [ ]:
run("python s1_peak_filter.py", cwd=RS); print(pd.read_csv(f"{RS}/out/peak_filter_stats.csv").to_string(index=False))

## 10.2 The motif loci each cell line carries
One table per cell line: every gene-motif copy found in that line's own promoter windows. The promoter windows differ between lines, so the five tables are different sizes and are **not** intersected.

In [ ]:
rows = []
for c in CELLS:
    L = pd.read_parquet(f"{RS}/out/loci/{c}.parquet")
    rows.append(dict(cell=c, loci=len(L), genes=L.gene_idx.nunique(), motifs=L.motif_id.nunique()))
print(pd.DataFrame(rows).to_string(index=False))
print("\ndistinct (gene, motif) pairs over all five lines:",
      f"{pd.concat([pd.read_parquet(f'{RS}/out/loci/{c}.parquet')[['gene_idx','motif_id']] for c in CELLS]).drop_duplicates().shape[0]:,}")

## 10.3 Which candidate readers bind, cell line by cell line (minutes)
`s3c_reader_binding_percell.py` intersects each line's own loci with the top peaks of every TF that JASPAR lists as a candidate reader of that motif, and keeps one record per (cell line, gene, motif, reader). It also records which TFs have a ChIP-seq experiment at all in each line (`assayed_tfs.json`) -- only readers measured in **both** lines of a pair are compared later, so a factor that was never assayed in one line can never look like a switch.

In [ ]:
run("python s3c_reader_binding_percell.py", cwd=RS)
b = pd.read_parquet(f"{RS}/out/reader_binding_percell.parquet")
i = pd.read_parquet(f"{RS}/out/interrogated_percell.parquet")
asy = json.load(open(f"{RS}/out/assayed_tfs.json"))
print(f"{len(b):,} (cell, gene, motif, reader) records over {b.cell.nunique()} cell lines; "
      f"{len(i):,} (cell, gene, motif) loci scanned")
print("\nTFs with a ChIP-seq experiment per cell line:", {k: len(v) for k, v in asy.items()})
for f in ("reader_binding_percell.parquet", "interrogated_percell.parquet"):
    pub = f"{WORK}/published_{f}"
    if os.path.exists(pub):
        new, old = pd.read_parquet(f"{RS}/out/{f}"), pd.read_parquet(pub)
        print(f"{f}: {len(new):,} records, published {len(old):,}",
              "| identical" if len(new) == len(old) else "| differs (see the note in 10.4)")

## 10.4 Fig. 5b and 5c

**Panel b** draws one grid per motif: rows are candidate readers, columns the five cell lines. Each bubble carries two numbers with the same numerator -- the number of that motif's bound genes at which this reader binds in this line -- over two different denominators:

| channel | meaning |
|---|---|
| **bubble size** | that count divided by the number of genes where *any* candidate reader binds the motif in that line, i.e. the share of the motif's bound genes this reader covers (breadth) |
| **bubble colour** | that count divided by the summed counts of all candidate readers of the motif in that line, i.e. this reader's share of all reader binding at the motif |
| **dark ring** | the **dominant reader**: the one bound at the most genes in that line, and at least `MIN_GENES = 20` of them |

No ChIP signal intensity enters either channel; both are counts of genes.

**Panel c** is the switch rate. For one motif and one pair of cell lines, a gene counts if (i) the motif copy is present in both lines' own promoter windows, (ii) at least two of the motif's candidate readers have ChIP data in both lines, and (iii) at least one of them is bound in each line. The gene **switches** when the two lists of bound readers are not identical -- a reader bound in one line only, a reader replaced by a different one, or both. The bar is the switched count divided by the qualifying count, **summed over all ten cell-line pairs** (not averaged), which is why its denominator counts gene-pair comparisons rather than distinct genes. The dots beside a bar give the number of distinct dominant readers that motif has across the five lines.

In [ ]:
run("python fig5bc_popularity_open.py", cwd=RS)
display(Image(f"{RS}/out/fig5bc_popularity_open.png", width=720))
sw = pd.read_csv(f"{RS}/out/per_motif_setswitch_open.csv")
print(sw.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
pub = f"{WORK}/published_per_motif_setswitch_open.csv"
if os.path.exists(pub):
    old = pd.read_csv(pub)
    same = sw.round(6).equals(old.round(6))
    print("\nidentical to the published table:", same)
    if not same:
        print("  ENCODE releases further TF experiments over time; a reader that was not assayed at publication can")
        print("  add gene comparisons to a pair. Regenerated beside published:")
        print(sw.merge(old, on="motif_id", suffixes=("", "_pub")).round(2).to_string(index=False))